# 1. Opis podataka

- Koriscene tabele: races, results, drivers.
- Results tabela - svaki red jedan vozac, jedna trka
- Instanca: vozač u trci (analitički deo) / vozač u sezoni (prediktivni deo).
- Bitne kolone u results: 
- grid - startana pozicija na trci 
- position - finalni plasman u trci po zavrsetku trke - moze biti string 
- positionOrder - uvek broj
- year
- driverId
- raceId
- Tabele drivers i races - spajanje, prikaz na grafikonu


In [ ]:
import pandas as pd

# Ucitavanje podataka
drivers = pd.read_csv('podaci/drivers.csv')
races = pd.read_csv('podaci/races.csv')
results = pd.read_csv('podaci/results.csv')

print(f"Broj redova u results: {results.shape[0]}")
print(f"Kolone: {results.columns.tolist()}")

Broj redova u results: 26759
Kolone: ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']


# 2. Čišćenje podataka

- Kako start utice na rezultat?
- grid - startana pozicija na trci 
- position/positionOrder - konacna pozicija u trci - Ukloniti DNF (Did Not Finish) vozače , DNf - na osnovu status id?
- Missing position - prazno ili '\N'
- Provera nedostajuće vrednosti.
- Konvertujem tipove podataka - ML algortimi rade sa numerickim vrednostima
- positionOrder - konacni plasaman kao broj (bolji za analizu mozda) - npr. klasifikovan iako je vrednost za poziciju prazna


In [7]:
# Uklanjanje DNF i nedostajućih vrednosti
results['position'] = pd.to_numeric(results['position'], errors='coerce')
results['grid'] = pd.to_numeric(results['grid'], errors='coerce')
results_clean = results[results['position'].notna() & (results['grid'] > 0)]

print(f"Broj redova posle čišćenja: {results_clean.shape[0]}")
print(f"Nedostajuće vrednosti po koloni:\n{results_clean.isnull().sum()}")


Broj redova posle čišćenja: 15746
Nedostajuće vrednosti po koloni:
resultId           0
raceId             0
driverId           0
constructorId      0
number             0
grid               0
position           0
positionText       0
positionOrder      0
points             0
laps               0
time               0
milliseconds       0
fastestLap         0
rank               0
fastestLapTime     0
fastestLapSpeed    0
statusId           0
dtype: int64


# 3. Kreiranje novih atributa (Feature engineering)

- results_clean - nova tabela (novi, csv)
- change_position - nova kolona, mera napretka ili pada u odnosu na start poziciju
- Promena change_position = grid – position (logika: npr. vozac startuje na 10, a zavrsio je na 4 poziciji, gleda se da li je napredovao i za koliko mesta)
- podium: nova kolona
- Indikator podijuma: 1 ako je vozač završio na poziciji 1-3, ako nije uzima 0
- Broj podijuma u prvih 12 trka


In [8]:
# Promena pozicije
results_clean['position_change'] = results_clean['grid'] - results_clean['position']

# Indikator podijuma
results_clean['podium'] = results_clean['position'].apply(lambda x: 1 if x in [1,2,3] else 0)

# Broj podijuma u prvih 12 trka po vozaču i sezoni
results_clean = results_clean.merge(races[['raceId', 'year']], on='raceId', how='left')
results_clean['race_number'] = results_clean.groupby(['year']).cumcount() + 1

def count_podiums(row):
    mask = (results_clean['year'] == row['year']) & (results_clean['driverId'] == row['driverId']) & (results_clean['race_number'] < row['race_number'])
    return results_clean.loc[mask, 'podium'].sum()

results_clean['podiums_before'] = results_clean.apply(count_podiums, axis=1)


C:\Users\Miroslav\AppData\Local\Temp\ipykernel_24524\83049289.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_clean['position_change'] = results_clean['grid'] - results_clean['position']
C:\Users\Miroslav\AppData\Local\Temp\ipykernel_24524\83049289.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_clean['podium'] = results_clean['position'].apply(lambda x: 1 if x in [1,2,3] else 0)


# 4. Eksplorativna analiza (EDA)

- Grafikon 1: Startna vs finalna pozicija (scatter)
- Grafikon 2: Distribucija promena pozicija (histogram)
- Grafikon 3: Broj podijuma šampioni vs ostali
- Prikaži neravnotežu klase (šampioni vs ostali)


In [10]:
import matplotlib.pyplot as plt
import seaborn as sns

# Grafikon 1: Startna vs finalna pozicija
plt.figure(figsize=(10,6))
plt.scatter(results_clean['grid'], results_clean['position'], alpha=0.2)
plt.xlabel('Startna pozicija (grid)')
plt.ylabel('Finalna pozicija')
plt.title('Startna vs finalna pozicija vozača')
plt.show()

# Grafikon 2: Distribucija promena pozicija
plt.figure(figsize=(10,6))
plt.hist(results_clean['position_change'], bins=30, edgecolor='black')
plt.xlabel('Promena pozicije (grid - position)')
plt.ylabel('Broj vozača')
plt.title('Distribucija promena pozicija')
plt.show()


ModuleNotFoundError: No module named 'matplotlib'

In [11]:
# Grafikon 3: Broj podijuma šampioni vs ostali
# Pretpostavljam da imaš kolonu is_champion (0/1) – ako ne, možeš je kreirati iz standings
# Ovde samo primer:

# results_clean['is_champion'] = ... # Dodaj iz standings
champion_podiums = results_clean[results_clean['is_champion'] == 1]['podium']
other_podiums = results_clean[results_clean['is_champion'] == 0]['podium']

plt.figure(figsize=(10,6))
plt.hist([other_podiums, champion_podiums], bins=10, label=['Ostali', 'Šampioni'], alpha=0.7)
plt.xlabel('Broj podijuma')
plt.ylabel('Broj vozača')
plt.title('Broj podijuma: šampioni vs ostali')
plt.legend()
plt.show()

# Neravnoteža klase
print('Broj šampiona:', results_clean['is_champion'].sum())
print('Broj ostalih:', (results_clean['is_champion'] == 0).sum())


KeyError: 'is_champion'

# 5. Formiranje prediktivnog skupa

Instanca = vozač u sezoni
Ulaz = rezultati iz prvih 12 trka
Cilj = da li je šampion


In [12]:
# Formiranje prediktivnog skupa
# Agregacija rezultata iz prvih 12 trka po vozaču i sezoni
pred_data = results_clean[results_clean['race_number'] <= 12].groupby(['year', 'driverId']).agg({
    'grid': 'mean',
    'position': 'mean',
    'podiums_before': 'max',
    'podium': 'sum',
    'position_change': 'mean',
    'is_champion': 'max'
}).reset_index()

print(pred_data.head())


KeyError: "Column(s) ['is_champion'] do not exist"

# 6. Primena jednog modela (baseline)

Izabrana metoda: Logistic Regression (jednostavan, interpretabilan, baseline)


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

X = pred_data[['grid', 'position', 'podiums_before', 'podium', 'position_change']]
y = pred_data['is_champion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Precision:', precision_score(y_test, y_pred, zero_division=0))
print('Recall:', recall_score(y_test, y_pred, zero_division=0))
print('F1 score:', f1_score(y_test, y_pred, zero_division=0))


ModuleNotFoundError: No module named 'sklearn'

# 7. Evaluacija

Koristim Precision, Recall, F1 score.
Ne koristim accuracy jer je dataset neuravnotežen (šampiona ima malo).


# 8. Zaključci

- Startna pozicija ima uticaj na finalni plasman, ali postoje značajna napredovanja i padovi.
- Rana forma (broj podijuma u prvih 12 trka) je dobar prediktor za šampiona.
- Dataset je neuravnotežen – evaluacija mora koristiti Precision, Recall, F1.
- Logistic Regression je dobar baseline, ali treba probati naprednije modele i dodatne analize.
